# Applied Economics Project: Statistical Data Analytics for Macroeconomic Indicators

**Based on:** Steinkamp, *Python for Engineering and Scientific Computing*, Chapter 9 — Statistical Computations

## Project Overview
In manufacturing, engineers monitor workpiece dimensions with statistical process control (SPC) so that
defects are caught *before* they happen. In this project you will apply the **same statistical toolkit**
(location parameters, dispersion parameters, frequency distributions, the normal distribution, skewness,
regression analysis, and control charts) to **macroeconomic data**:

| Manufacturing concept (book) | Economics analogue (this project) |
|---|---|
| Gear-shaft diameter measurements | Monthly inflation rate (%) |
| Workpiece sample (n = 5 per hour) | Regional unemployment sample (n = 5 regions per quarter) |
| Tolerance band on a technical drawing | Central bank inflation target band |
| Machine capability index Cm, Cmk | "Monetary policy capability" index |
| Roughness depth / resistance value | Monthly stock market returns |
| Relative humidity → fiber moisture (regression) | Inflation rate → unemployment rate (Phillips curve) |
| Quality control chart (SPC) | Regional unemployment control chart |

### Learning goals
1. Simulate realistic macroeconomic time series with NumPy.
2. Persist and reload data from disk (as a real statistical office would).
3. Compute and interpret location parameters (mean, median, mode, harmonic & geometric mean).
4. Compute and interpret dispersion parameters (span, standard deviation) and build a
   "policy capability index" analogous to Cm / Cmk.
5. Build frequency tables / histograms and connect them to the normal distribution.
6. Use `scipy.integrate.quad` (or `scipy.stats.norm`) to compute probabilities under the normal curve.
7. Compute skewness and interpret tail risk in financial returns.
8. Run a linear regression (Phillips curve) with `scipy.stats.linregress` and interpret r, m, a.
9. Build a two-lane control chart to monitor regional unemployment dispersion over time.

> Work through the notebook section by section. Each section has a short **theory recap**, followed by a
> code cell with `# TODO` markers for you to complete. Consult the accompanying **cheat sheet** if you
> get stuck.

## 0. Setup

Import the libraries you will need throughout the project. This mirrors Listing 9.1–9.17 of the book,
where `numpy`, `scipy.stats`, `scipy.integrate`, and `matplotlib.pyplot` form the core toolkit.

In [ ]:
# TODO: import numpy as np, pandas as pd, matplotlib.pyplot as plt,
# scipy.stats as stats, and quad from scipy.integrate

# TODO: set a random seed (e.g. np.random.seed(42)) for reproducible results


## 1. Generating Simulated Macroeconomic Data

Just as Listing 9.1 uses `np.random.normal(setpoint, s, size=n)` to simulate a gear-shaft diameter, we
simulate **60 months** of three indicators for a fictitious country "Economland":

* **Inflation rate** — normally distributed around a central-bank target of **3.0 %** with a standard
  deviation of **0.5 %**.
* **Stock market monthly return** — normally distributed around **0.8 %** with a standard deviation of
  **4.0 %** (returns can be negative).
* **Unemployment rate** — follows a simplified **Phillips-curve** relationship with inflation:
  `unemployment = 8 - 1.5 * inflation + noise`, where `noise ~ N(0, 0.6)`.

Round all values to 2 decimal places (as in Listing 9.1, `np.around(values, decimals=2)`).

In [ ]:
n = 60  # months

# TODO: generate `inflation`: 60 normally distributed values, mean 3.0, std 0.5
#       round to 2 decimals with np.around()

# TODO: generate `stock_returns`: 60 normally distributed values, mean 0.8, std 4.0
#       round to 2 decimals

# TODO: generate `noise`: 60 normally distributed values, mean 0, std 0.6
# TODO: compute `unemployment = 8 - 1.5 * inflation + noise`, round to 2 decimals

# TODO: build a pandas DataFrame `df` with columns: month, inflation, unemployment, stock_return
# TODO: print df.head() and the type of the inflation array


## 2. Persisting and Reloading the Data

In real life, statistical offices store measurement series on disk. Listing 9.3 / 9.4 use
`np.savetxt()` and `np.loadtxt()`. Here, save the full DataFrame to a CSV file with `df.to_csv()`, then
reload it with `pd.read_csv()` and confirm the reloaded data matches the original.

In [ ]:
# TODO: save `df` to "economland_data.csv" with df.to_csv(), index=False

# TODO: reload it into `df_loaded` with pd.read_csv()
# TODO: print the number of rows loaded and df_loaded.head()

# TODO: assert that df["inflation"] matches df_loaded["inflation"] (np.allclose)


## 3. Frequency Distribution of the Inflation Rate

Following Section 9.2 of the book: determine the number of classes
$k = \lceil \sqrt{n} \rceil$, the span $R = x_{max}-x_{min}$, and the class interval $w = R/k$.
Then build a frequency table with `np.histogram()` and visualize it with `ax.hist()`.

In [ ]:
values = df_loaded["inflation"].values
n = len(values)

# TODO: compute k = number of classes = round(sqrt(n)) using int(np.sqrt(n)+0.5)
# TODO: compute minimum, maximum with np.amin / np.amax
# TODO: compute span R = maximum - minimum (round to 2 decimals)
# TODO: compute class interval w = R / k (round to 2 decimals)

# TODO: compute H, I = np.histogram(values, bins=k)
# TODO: compute relative frequency h = 100*H/n

# TODO: print minimum, maximum, span, k, w, H, h

# TODO: plot a histogram of `values` with k bins using ax.hist(); label the axes


## 4. Location Parameters of the Inflation Rate

Section 9.3 of the book covers four location parameters. Because inflation values here are all
positive, we can safely compute all four:

$$\bar{x}=\frac{1}{n}\sum_{i=1}^n x_i \qquad
\bar{x}_{harmonic} = n\left(\sum_{i=1}^n \frac{1}{x_i}\right)^{-1} \qquad
\bar{x}_{geometric} = \sqrt[n]{x_1 \cdot x_2 \cdots x_n}$$

Use `np.mean()`, `np.median()`, `scipy.stats.mode()`, `scipy.stats.hmean()`, and `scipy.stats.gmean()`.

In [ ]:
# TODO: compute the arithmetic mean of `values` with np.mean()
# TODO: compute the median with np.median()
# TODO: compute the mode with stats.mode(values, keepdims=True)
# TODO: compute the harmonic mean with stats.hmean()
# TODO: compute the geometric mean with stats.gmean()

# TODO: print all five results with clear labels


## 5. Dispersion Parameters & a "Monetary-Policy Capability Index"

Section 9.4 of the book computes a *machine capability index* $C_m = T/(6s) \ge 1.67$ to check whether a
machine can reliably stay within a tolerance band. We build the economic analogue:

Suppose the central bank's official **inflation target band** is **[1.5 %, 4.5 %]** (tolerance
$T = 4.5-1.5 = 3.0$). Compute:

* the standard deviation $s$ of the inflation series (`np.std(values, ddof=1)`)
* the **capability index** $C_m = T / (6s)$ — analogous to machine capability: is monetary policy
  "capable" of consistently hitting the target band? ($C_m \ge 1.67$ is considered good.)
* the **centering index** $C_{mk} = \Delta_{krit} / (3s)$, where $\Delta_{krit}$ is the smaller of
  `(upper_limit - mean)` and `(mean - lower_limit)` — i.e., is the *average* inflation rate centered in
  the band, or is it drifting toward one edge?

In [ ]:
# TODO: compute the standard deviation s of `values` with ddof=1
lower_limit, upper_limit = 1.5, 4.5

# TODO: compute the tolerance T = upper_limit - lower_limit
# TODO: compute Cm = T / (6*s)

# TODO: compute delta_o = upper_limit - mw  (mw = arithmetic mean from Section 4)
# TODO: compute delta_u = mw - lower_limit
# TODO: compute delta_k = the smaller of delta_o and delta_u
# TODO: compute Cmk = delta_k / (3*s)

# TODO: print s, Cm, Cmk
# TODO: print a one-line interpretation: is Cm >= 1.67 and Cmk >= 1.67?


## 6. The Normal Distribution & Probability of Staying On-Target

Section 9.5 fits a Gaussian density $g(x)$ to the data and uses `scipy.integrate.quad` to compute the
probability that a value falls in a given range. Here we:

1. Plot the fitted normal density $g(x)$ for the inflation series (using the sample mean and std as
   $\mu$ and $\sigma$).
2. Compute the probability that inflation is **within the target band [1.5%, 4.5%]** by integrating the
   density over that range with `quad()`.
3. Compute (for comparison) the classic $\pm\sigma,\ \pm2\sigma,\ \pm3\sigma$ probabilities
   (68.27 %, 95.45 %, 99.73 %).

In [ ]:
def g(x, sigma, mu):
    # TODO: implement the Gaussian density function g(x) = exp(-0.5*(x-mu)**2/sigma**2)/(sigma*sqrt(2*pi))
    pass

mu, sigma = mw, s   # from Sections 4 and 5

# TODO: build an x array from mu-4*sigma to mu+4*sigma with step 0.01, and compute y = g(x, sigma, mu)
# TODO: plot the density curve; add vertical reference lines at lower_limit and upper_limit

# TODO: use quad() to integrate g between lower_limit and upper_limit -> probability of staying on target
# TODO: use quad() to compute probabilities within ±1σ, ±2σ, ±3σ (classic 68/95/99.7 rule)

# TODO: print all four probabilities as percentages


## 7. Skewness of Stock Market Returns

Section 9.6 introduces skewness as a measure of asymmetry. Financial returns are often **negatively
skewed** (large crashes are more extreme than large rallies). Compute:

* Pearson's approximate skew: $S_1 = (\bar{x}-\tilde{x})/s$
* The exact skew via `scipy.stats.skew()`

and interpret the sign in terms of **tail risk**.

In [ ]:
returns = df_loaded["stock_return"].values

# TODO: compute mean, median, std (ddof=1) of `returns`
# TODO: compute Pearson's approximate skew S1 = (mean - median) / std
# TODO: compute the exact skew S2 with stats.skew(returns)

# TODO: print mean, median, std, S1, S2
# TODO: print an interpretation: if S2 < 0 -> right-skewed (crash risk); else left-skewed


## 8. Regression Analysis: The Phillips Curve

Section 9.7 regresses moisture content on humidity. Here we regress **unemployment (Y)** on
**inflation (X)** — the textbook Phillips-curve trade-off. Use `scipy.stats.linregress()` to obtain the
slope $m$, intercept $a$, and correlation coefficient $r$, then plot the scatter plot with the fitted
regression line.

In [ ]:
X = df_loaded["inflation"].values
Y = df_loaded["unemployment"].values

# TODO: run m, a, r, p, e = stats.linregress(X, Y)

# TODO: print slope, intercept, correlation coefficient, and the regression equation

# TODO: create a scatter plot of X vs Y ('rx' markers) and overlay the regression line (X, m*X+a)
# TODO: label axes and title, add a legend

# TODO: print a one-line interpretation of r (strong negative / positive / weak correlation)


## 9. Project Task: A Regional Unemployment Control Chart

Mirroring Section 9.8, imagine the national statistics office samples the unemployment rate in
**5 regions every quarter** for **10 quarters** and wants a two-lane control chart (mean chart +
standard-deviation chart) to spot regions/quarters where dispersion across regions becomes abnormal
(a signal of diverging regional economies needing policy attention).

Use factors for a sample size of n = 5 (same as the book): $A_3 = 1.152$, $B_4 = 1.669$.

$$UCL_{\bar x} = \bar{\bar x} + A_3 \bar s \qquad LCL_{\bar x} = \bar{\bar x} - A_3 \bar s \qquad
UCL_s = B_4 \bar s$$

In [ ]:
rows, columns = 5, 10          # 5 regions, 10 quarters
A3, B4 = 1.152, 1.669

# TODO: simulate `regional_data`: rows*columns normally distributed values around a national_target
#       of 6.0 with std 0.9; round to 2 decimals
# TODO: reshape into a (rows, columns) table with order='F' (Fortran order, as in Listing 9.16)

quarter = np.arange(1, columns + 1)
mw_q, staw_q = [], []
# TODO: loop i in range(columns); for each quarter compute the mean and std (ddof=1) of
#       table[0:rows, i] using slicing (no inner for-loop needed); append rounded values
#       to mw_q and staw_q

# TODO: compute mmw = mean of mw_q, mws = mean of staw_q
# TODO: compute UCLm = mmw + A3*mws, LCLm = mmw - A3*mws, UCLs = B4*mws

# TODO: print the table, quarterly means, quarterly std devs, and the three limits

# TODO: plot a two-panel figure (plt.subplots(2,1)):
#   panel 0: mean chart -> UCL (red), center line mmw (green), LCL (red), and the mw_q polyline (blue 'x-')
#   panel 1: std-dev chart -> UCL (red) and the staw_q polyline (green 'x-')
#   label axes and titles

# TODO: identify and print any quarters where the quarterly mean exceeds UCLm or falls below LCLm


## 10. Conclusions & Reflection

Answer the following in a markdown cell:

1. Based on Section 5's capability indices, is Economland's central bank reliably hitting its inflation
   target? Explain using $C_m$ and $C_{mk}$.
2. Based on Section 6, what is the probability that inflation stays inside the target band in any given
   month? Is this an acceptable risk for policymakers?
3. Based on Section 7, does the stock market in this simulation show crash risk (negative skew) or rally
   risk (positive skew)?
4. Based on Section 8, do these data support a Phillips-curve trade-off? What is the economic
   interpretation of the slope $m$?
5. Based on Section 9, were there any quarters where regional unemployment dispersion signaled a
   need for policy intervention?

*(This is a synthetic dataset for practicing the statistical toolkit — repeat the analysis with real
data, e.g., from FRED, Eurostat, or the World Bank, as a stretch goal.)*

*(Write your answers here.)*